Name: Sufiyan Shaikh

Roll No:SCFP125021

# Lab 5 — Workflow Design with LLMs

Run cells from top to bottom in VS Code. This shorter notebook covers all six exercises using LangChain prompt templates and output parsers.

- Default: `USE_LLM = False` uses sample responses to test the workflow without an API key.
- For actual LLM responses, set `OPENAI_API_KEY` in your environment and change `USE_LLM = True`

## 1. Install libraries


In [1]:
pip install -q -U langchain langchain-core langchain-openai pydantic requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Imports and helper functions


In [2]:
import os, json, requests
from typing import List
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from langchain_core.messages import AIMessage
from langchain_core.runnables import RunnableLambda

USE_LLM = False
if USE_LLM:
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    if not os.getenv("OPENAI_API_KEY"):
        raise ValueError("Set OPENAI_API_KEY before using live mode.")

def model_for(sample):
    return llm if USE_LLM else RunnableLambda(lambda _: AIMessage(content=json.dumps(sample) if isinstance(sample, dict) else sample))

def structured_chain(prompt, schema, sample, chat=False):
    parser = PydanticOutputParser(pydantic_object=schema)
    if isinstance(prompt, PromptTemplate):
        prompt = prompt.partial(format_instructions=parser.get_format_instructions())
    else:
        prompt = prompt.partial(format_instructions=parser.get_format_instructions())
    return prompt | model_for(sample) | parser

def show(title, result):
    print("\n---", title, "---")
    print(result.model_dump() if hasattr(result, "model_dump") else result)

## Exercise 1 — Customer Review to Support Ticket


In [3]:
class Complaint(BaseModel):
    complaint: str
    product_or_feature: str
    sentiment: str

p1 = PromptTemplate.from_template(
    "Extract complaint, product_or_feature, sentiment. Don't invent facts.\n{format_instructions}\nReview: {review}"
)
c1 = structured_chain(p1, Complaint, {
    "complaint":"App crashes during photo upload",
    "product_or_feature":"Mobile photo upload",
    "sentiment":"Negative"
})
data1 = c1.invoke({"review":"The app crashes every time I upload a photo. Very frustrating."})

p2 = ChatPromptTemplate.from_messages([
    ("system","You are a support ticket writer. Use only the structured data."),
    ("human","Write a short internal ticket from: {data}")
])
ticket = (p2 | model_for("Issue: App crashes during photo upload. Sentiment: Negative. Investigate the mobile photo-upload feature.") | StrOutputParser()).invoke({"data":data1.model_dump_json()})
show("Step 1", data1); show("Step 2", ticket)


--- Step 1 ---
{'complaint': 'App crashes during photo upload', 'product_or_feature': 'Mobile photo upload', 'sentiment': 'Negative'}

--- Step 2 ---
Issue: App crashes during photo upload. Sentiment: Negative. Investigate the mobile photo-upload feature.


## Exercise 2 — Research Abstract to Simple Summary

In [4]:
class Research(BaseModel):
    research_question: str
    method: str
    key_finding: str

p1 = PromptTemplate.from_template(
    "Extract research_question, method, key_finding.\n{format_instructions}\nAbstract: {abstract}"
)
data2 = structured_chain(p1, Research, {
    "research_question":"Does spaced practice improve memory?",
    "method":"Compared spaced practice with one-session practice",
    "key_finding":"Spaced practice improved later recall"
}).invoke({"abstract":"The study compared spaced and massed vocabulary practice. The spaced group recalled more words later."})

p2 = ChatPromptTemplate.from_messages([
    ("system","Explain science to a general audience in simple language."),
    ("human","Summarize these details: {data}")
])
summary = (p2 | model_for("Practicing over several sessions may help people remember more than studying everything at once.") | StrOutputParser()).invoke({"data":data2.model_dump_json()})
show("Step 1", data2); show("Step 2", summary)


--- Step 1 ---
{'research_question': 'Does spaced practice improve memory?', 'method': 'Compared spaced practice with one-session practice', 'key_finding': 'Spaced practice improved later recall'}

--- Step 2 ---
Practicing over several sessions may help people remember more than studying everything at once.


## Exercise 3 — Interview Transcript to Scorecard

In [5]:
class Answer(BaseModel):
    question: str
    answer: str
class Answers(BaseModel):
    answers: List[Answer]
class Skill(BaseModel):
    skill: str
    demonstrated: bool
    evidence_or_gap: str
class Skills(BaseModel):
    evaluations: List[Skill]
class Row(BaseModel):
    skill: str
    demonstrated: bool
    evidence_or_gap: str
    rating: str
class Scorecard(BaseModel):
    scorecard_rows: List[Row]
    overall_recommendation: str

transcript = "Q: Problem solved? A: I removed duplicate records and tested validation. Q: Teamwork? A: I share tasks and progress."
a_prompt = PromptTemplate.from_template("Extract question-answer pairs.\n{format_instructions}\nTranscript: {transcript}")
a_data = structured_chain(a_prompt, Answers, {"answers":[
    {"question":"Problem solved?","answer":"Removed duplicate records and tested validation."},
    {"question":"Teamwork?","answer":"Shares tasks and progress."}
]}).invoke({"transcript":transcript})

s_prompt = ChatPromptTemplate.from_messages([
    ("system","You are a technical interviewer. Assess only evidence and flag gaps."),
    ("human","Assess skills {skills} using answers {data}.\n{format_instructions}")
])
s_data = structured_chain(s_prompt, Skills, {"evaluations":[
    {"skill":"Problem solving","demonstrated":True,"evidence_or_gap":"Removed duplicate records and tested validation."},
    {"skill":"Teamwork","demonstrated":True,"evidence_or_gap":"Described sharing tasks and progress."},
    {"skill":"Python","demonstrated":False,"evidence_or_gap":"No Python evidence in transcript."}
]}).invoke({"skills":"Problem solving, Teamwork, Python","data":a_data.model_dump_json()})

f_prompt = ChatPromptTemplate.from_messages([
    ("system","You are a hiring panel coordinator. Be fair and evidence-based."),
    ("human","Create scorecard from {data}.\n{format_instructions}")
])
final3 = structured_chain(f_prompt, Scorecard, {"scorecard_rows":[
    {"skill":"Problem solving","demonstrated":True,"evidence_or_gap":"Duplicate removal and validation test","rating":"Strong"},
    {"skill":"Teamwork","demonstrated":True,"evidence_or_gap":"Shared tasks and progress","rating":"Partial"},
    {"skill":"Python","demonstrated":False,"evidence_or_gap":"No evidence provided","rating":"Not demonstrated"}
],"overall_recommendation":"Conduct a follow-up Python interview; current evidence is incomplete."}).invoke({"data":s_data.model_dump_json()})
show("Step 1", a_data); show("Step 2", s_data); show("Step 3", final3)


--- Step 1 ---
{'answers': [{'question': 'Problem solved?', 'answer': 'Removed duplicate records and tested validation.'}, {'question': 'Teamwork?', 'answer': 'Shares tasks and progress.'}]}

--- Step 2 ---
{'evaluations': [{'skill': 'Problem solving', 'demonstrated': True, 'evidence_or_gap': 'Removed duplicate records and tested validation.'}, {'skill': 'Teamwork', 'demonstrated': True, 'evidence_or_gap': 'Described sharing tasks and progress.'}, {'skill': 'Python', 'demonstrated': False, 'evidence_or_gap': 'No Python evidence in transcript.'}]}

--- Step 3 ---
{'scorecard_rows': [{'skill': 'Problem solving', 'demonstrated': True, 'evidence_or_gap': 'Duplicate removal and validation test', 'rating': 'Strong'}, {'skill': 'Teamwork', 'demonstrated': True, 'evidence_or_gap': 'Shared tasks and progress', 'rating': 'Partial'}, {'skill': 'Python', 'demonstrated': False, 'evidence_or_gap': 'No evidence provided', 'rating': 'Not demonstrated'}], 'overall_recommendation': 'Conduct a follow-up

## Exercise 4 — Event Plan and Weather Recommendation

In [6]:
class Event(BaseModel):
    event_type: str
    event_date: str
    location: str
    indoor_outdoor_preference: str
class Weather(BaseModel):
    temperature: float
    precipitation_chance: int
    conditions: str

ep = PromptTemplate.from_template("Extract event_type, event_date (YYYY-MM-DD), location, indoor_outdoor_preference.\n{format_instructions}\nDescription: {description}")
event = structured_chain(ep, Event, {
    "event_type":"Community sports day","event_date":"2026-10-25",
    "location":"Pune, India","indoor_outdoor_preference":"Outdoor, indoor backup available"
}).invoke({"description":"Outdoor community sports day in Pune on 25 October 2026, with an indoor hall as backup."})

# Format the API query from the extracted fields.
weather_query = PromptTemplate.from_template("Weather API query for {location} on {event_date}")
print(weather_query.format(location=event.location, event_date=event.event_date))

def get_weather(e):
    try:
        g = requests.get("https://geocoding-api.open-meteo.com/v1/search",
            params={"name":e.location.split(",")[0],"count":1}, timeout=10)
        g.raise_for_status(); place = g.json()["results"][0]
        r = requests.get("https://api.open-meteo.com/v1/forecast", params={
            "latitude":place["latitude"],"longitude":place["longitude"],
            "daily":"temperature_2m_max,precipitation_probability_max,weather_code",
            "timezone":"auto","start_date":e.event_date,"end_date":e.event_date}, timeout=10)
        r.raise_for_status(); d = r.json()["daily"]
        return Weather(temperature=d["temperature_2m_max"][0],
            precipitation_chance=d["precipitation_probability_max"][0] or 0,
            conditions="Weather code "+str(d["weather_code"][0]))
    except Exception as err:
        print("Weather API unavailable/date unsupported; using sample data:", err)
        return Weather(temperature=27, precipitation_chance=35, conditions="Partly cloudy (sample)")
w = get_weather(event)

rp = ChatPromptTemplate.from_messages([
    ("system","You are an event planning assistant. Use the event and weather data; don't overstate certainty."),
    ("human","Event: {event}\nWeather: {weather}\nRecommend proceed, reschedule, or move indoors.")
])
rec = (rp | model_for("Keep the indoor backup ready because precipitation is possible. Recheck the forecast before the event.") | StrOutputParser()).invoke({
    "event":event.model_dump_json(),"weather":w.model_dump_json()
})
show("Step 1", event); show("Step 2", w); show("Step 3", rec)

Weather API query for Pune, India on 2026-10-25

--- Step 1 ---
{'event_type': 'Community sports day', 'event_date': '2026-10-25', 'location': 'Pune, India', 'indoor_outdoor_preference': 'Outdoor, indoor backup available'}

--- Step 2 ---
{'temperature': 32.9, 'precipitation_chance': 10, 'conditions': 'Weather code 3'}

--- Step 3 ---
Keep the indoor backup ready because precipitation is possible. Recheck the forecast before the event.


## Exercise 5 — Support Chat to Escalation Report

In [7]:
class Support(BaseModel):
    issue: str
    customer_tone: str
    resolution_attempted: str
class Readiness(BaseModel):
    missing_fields: List[str]
    is_ready_to_escalate: bool

sp = PromptTemplate.from_template("Extract issue, customer_tone, resolution_attempted.\n{format_instructions}\nChat: {chat}")
support = structured_chain(sp, Support, {
    "issue":"Password reset email does not arrive",
    "customer_tone":"Frustrated",
    "resolution_attempted":"Checked spam and retried; issue remains"
}).invoke({"chat":"Customer cannot receive password reset email. Agent asked them to check spam and retry; still not working."})

gp = ChatPromptTemplate.from_messages([
    ("system","You are an escalation-readiness reviewer. Identify missing information."),
    ("human","Check this summary: {data}. Look for account ID, error code, and relevant time.\n{format_instructions}")
])
gaps = structured_chain(gp, Readiness, {
    "missing_fields":["account ID","error code","time of last attempt"],
    "is_ready_to_escalate":False
}).invoke({"data":support.model_dump_json()})

rp = ChatPromptTemplate.from_messages([
    ("system","You are a support operations lead. Write a prioritized concise report."),
    ("human","Summary: {summary}\nGap check: {gaps}\nWrite priority, issue, attempts, missing data, next steps.")
])
report = (rp | model_for("Priority: Medium. Password-reset email is not arriving despite checking spam and retrying. Collect account ID, error code if present, and time of last attempt. Then inspect email delivery logs and update the customer.") | StrOutputParser()).invoke({
    "summary":support.model_dump_json(),"gaps":gaps.model_dump_json()
})
show("Step 1", support); show("Step 2", gaps); show("Step 3", report)


--- Step 1 ---
{'issue': 'Password reset email does not arrive', 'customer_tone': 'Frustrated', 'resolution_attempted': 'Checked spam and retried; issue remains'}

--- Step 2 ---
{'missing_fields': ['account ID', 'error code', 'time of last attempt'], 'is_ready_to_escalate': False}

--- Step 3 ---
Priority: Medium. Password-reset email is not arriving despite checking spam and retrying. Collect account ID, error code if present, and time of last attempt. Then inspect email delivery logs and update the customer.


## Exercise 6 — Fitness Goal to Weekly Workout Plan

In [8]:
class Fitness(BaseModel):
    goal_type: str
    fitness_level: str
    days_available: int
    equipment_access: str
class WorkoutDay(BaseModel):
    day: str
    focus: str
    exercises: List[str]
    duration_minutes: int
    notes: str
class Plan(BaseModel):
    plan: List[WorkoutDay]

fp = PromptTemplate.from_template("Extract goal_type, fitness_level, days_available, equipment_access. Use days_per_week if needed.\n{format_instructions}\nGoal: {goal}\nDays per week: {days_per_week}")
fitness = structured_chain(fp, Fitness, {
    "goal_type":"Improve general fitness and stamina","fitness_level":"Beginner",
    "days_available":3,"equipment_access":"No equipment; bodyweight and walking"
}).invoke({"goal":"Beginner wanting better fitness and stamina; no gym equipment.","days_per_week":3})

wp = ChatPromptTemplate.from_messages([
    ("system","You are a certified personal trainer. Make a safe beginner plan, include rest days, and use only extracted details."),
    ("human","Fitness details: {data}\n{format_instructions}\nCreate a seven-day plan with the stated number of workout days.")
])
plan = structured_chain(wp, Plan, {"plan":[
    {"day":"Monday","focus":"Full body","exercises":["Warm-up walk","Chair squats 2x8","Wall push-ups 2x8"],"duration_minutes":20,"notes":"Comfortable pace."},
    {"day":"Tuesday","focus":"Rest","exercises":["Optional gentle walk"],"duration_minutes":10,"notes":"Recovery."},
    {"day":"Wednesday","focus":"Cardio and core","exercises":["Comfortable walk 15 minutes","Bird-dog 2x6 each side"],"duration_minutes":25,"notes":"Stop if you feel pain or dizziness."},
    {"day":"Thursday","focus":"Rest","exercises":["Rest"],"duration_minutes":0,"notes":"Recovery."},
    {"day":"Friday","focus":"Full body","exercises":["Warm-up walk","Chair squats 2x8","Wall push-ups 2x8"],"duration_minutes":20,"notes":"Keep movements controlled."},
    {"day":"Saturday","focus":"Optional recovery","exercises":["Easy walk"],"duration_minutes":15,"notes":"Optional only."},
    {"day":"Sunday","focus":"Rest","exercises":["Rest"],"duration_minutes":0,"notes":"Prepare for next week."}
]}).invoke({"data":fitness.model_dump_json()})
show("Step 1", fitness); show("Step 2", plan)


--- Step 1 ---
{'goal_type': 'Improve general fitness and stamina', 'fitness_level': 'Beginner', 'days_available': 3, 'equipment_access': 'No equipment; bodyweight and walking'}

--- Step 2 ---
{'plan': [{'day': 'Monday', 'focus': 'Full body', 'exercises': ['Warm-up walk', 'Chair squats 2x8', 'Wall push-ups 2x8'], 'duration_minutes': 20, 'notes': 'Comfortable pace.'}, {'day': 'Tuesday', 'focus': 'Rest', 'exercises': ['Optional gentle walk'], 'duration_minutes': 10, 'notes': 'Recovery.'}, {'day': 'Wednesday', 'focus': 'Cardio and core', 'exercises': ['Comfortable walk 15 minutes', 'Bird-dog 2x6 each side'], 'duration_minutes': 25, 'notes': 'Stop if you feel pain or dizziness.'}, {'day': 'Thursday', 'focus': 'Rest', 'exercises': ['Rest'], 'duration_minutes': 0, 'notes': 'Recovery.'}, {'day': 'Friday', 'focus': 'Full body', 'exercises': ['Warm-up walk', 'Chair squats 2x8', 'Wall push-ups 2x8'], 'duration_minutes': 20, 'notes': 'Keep movements controlled.'}, {'day': 'Saturday', 'focus': '